# PharmaShield GCC — Layer 3: AI Risk Engines (C2, C3, C4)
Group 3 · AI in Operations. Plan §4.3 and §5.2 steps 4–6. Run Layer 1 first (`notebooks/01_layer1_data_ingestion.ipynb`), then run all cells here top to bottom (plan step 10).

GenAI disclosure (plan §6.1): code generated with Claude and reviewed by the team; see `docs/GENAI_PROMPT_LOG_layer2_3.md`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'layer3_engines').exists())
sys.path.insert(0, str(ROOT / 'layer3_engines'))
import run_engines
from engines.io import P
pd.set_option('display.max_colwidth', 120)

## Run all three engines
Fills `C2_delivery_risk`, `C3_price_integrity`, `C4_compliance_risk` and `shap_top3_reasons` in `data/processed/suppliers_matrix.csv`.

In [2]:
run_engines.main(['--log-level', 'WARNING'])


=== Layer 3 engines summary ===
  C2 XGBoost  test ROC-AUC 0.797 | recall 0.92 | precision 0.20   (scale_pos_weight 19.6)
  C2 LogReg   test ROC-AUC 0.786 | recall 0.86 | precision 0.29   (baseline)
  C3 IsolationForest: 422 of 8436 scored lines flagged
  C4 letters  data_integrity       14/50 positive | OOF ROC-AUC 0.966
  C4 letters  sterility            24/50 positive | OOF ROC-AUC 0.994
  C4 letters  process_validation   16/50 positive | OOF ROC-AUC 0.79
  C4 letters  quality_unit         46/50 positive | OOF ROC-AUC 0.625
  C4 contracts LLM vs rules agreement: {'force_majeure': '11/11', 'lead_time': '11/11', 'late_delivery_penalty': '11/11', 'single_source': '11/11'}

          manufacturer_id  eligible_for_ranking  is_probation  C1_landed_cost  C2_delivery_risk  C3_price_integrity  C4_compliance_risk  C5_route_exposure
        ACCORD HEALTHCARE                  True          True             0.0            0.1033                 0.0              0.6402              0.272
       

0

## Step 4 — C2: XGBoost vs Logistic Regression (out-of-time test 2014–15)

In [3]:
m = json.loads((P.outputs / 'c2_metrics.json').read_text())
pd.DataFrame({'XGBoost': m['xgboost_test'], 'LogisticRegression': m['logistic_regression_test']}).drop(['confusion_matrix', 'threshold'])

,XGBoost,LogisticRegression
roc_auc,0.7975,0.7856
pr_auc,0.3238,0.3035
recall_late,0.9185,0.8596
precision_late,0.2012,0.2868
f1_late,0.3301,0.4301
f2_late,0.5362,0.6142


In [4]:
print('confusion matrix XGBoost:', m['xgboost_test']['confusion_matrix'])
print('confusion matrix LogReg :', m['logistic_regression_test']['confusion_matrix'])
pd.Series(m['global_feature_importance_mean_abs_shap'], name='mean |SHAP|')

confusion matrix XGBoost: {'tn': 891, 'fp': 1298, 'fn': 29, 'tp': 327}
confusion matrix LogReg : {'tn': 1428, 'fp': 761, 'fn': 50, 'tp': 306}


Delivery terms (Incoterm)    0.5521
Freight-to-value ratio       0.3735
Shipment weight              0.3258
Site delivery history        0.3113
Plant country                0.1114
Shipping mode                0.0984
Name: mean |SHAP|, dtype: float64

### Feature-based inference for every carboplatin maker (plan §3.1) with top-3 SHAP drivers

In [5]:
c2 = pd.read_csv(P.outputs / 'c2_panel_predictions.csv')
c2['top_driver'] = c2['shap_top3_reasons'].map(lambda s: '; '.join(f"{d['feature']} ({d['value']}): {d['effect']}" for d in json.loads(s)[:2]))
c2[['manufacturer_id', 'C2_delivery_risk', 'top_driver']]

,manufacturer_id,C2_delivery_risk,top_driver
0,ACCORD HEALTHCARE,0.1033,Freight-to-value ratio (not known for UAE shipments: averaged over real shipments from India): lowers late-delivery ...
1,APOTEX,0.1033,Freight-to-value ratio (not known for UAE shipments: averaged over real shipments from India): lowers late-delivery ...
2,AVYXA PHARMA,0.0263,Delivery terms (Incoterm) (not known for UAE shipments: averaged over real shipments from Switzerland): lowers late-...
3,BPI LABS,0.1033,Freight-to-value ratio (not known for UAE shipments: averaged over real shipments from India): lowers late-delivery ...
4,CIPLA,0.1142,Freight-to-value ratio (not known for UAE shipments: averaged over real shipments from India): lowers late-delivery ...
5,EUGIA US,0.1033,Freight-to-value ratio (not known for UAE shipments: averaged over real shipments from India): lowers late-delivery ...
6,FRESENIUS KABI USA,0.1033,Plant country (unknown (not stated on the drug label)): no data: neutral score = median of the makers whose plant co...
7,GLAND PHARMA,0.1033,Freight-to-value ratio (not known for UAE shipments: averaged over real shipments from India): lowers late-delivery ...
8,HERAEUS PRECIOUS METALS AND,0.0428,Delivery terms (Incoterm) (not known for UAE shipments: averaged over real shipments from Germany): lowers late-deli...
9,HETERO LABS,0.1135,Freight-to-value ratio (not known for UAE shipments: averaged over real shipments from India): lowers late-delivery ...


## Step 5 — C3: Isolation Forest billing anomalies (3 worked examples for slide 4)

In [6]:
pd.read_csv(P.outputs / 'c3_anomaly_examples.csv')[['id', 'vendor', 'product_group', 'why_flagged']]

,id,vendor,product_group,why_flagged
0,11079,SCMS from RDC,ARV,"$19,481 freight on a $58.75 line: freight = 331.6x the goods value (group median 0.06x); unit price $0.39 vs group m..."
1,60994,"Orgenics, Ltd",HRDT,$981 freight on a $8.00 line: freight = 122.6x the goods value (group median 0.12x); unit price $0.08 vs group media...
2,61882,Aurobindo Pharma Limited,ARV,"$4,117 freight on a $5.06 line: freight = 813.6x the goods value (group median 0.06x); unit price $0.04 vs group med..."


## Step 6 — C4: warning-letter classifier and contract clauses

In [7]:
c4m = json.loads((P.outputs / 'c4_metrics.json').read_text())
pd.DataFrame(c4m['warning_letter_classifier']).T

,positives,letters,method,oof_roc_auc,oof_f1_at_0.5
data_integrity,14,50,"TF-IDF + LogisticRegression, 5-fold OOF",0.966,0.783
sterility,24,50,"TF-IDF + LogisticRegression, 5-fold OOF",0.994,0.957
process_validation,16,50,"TF-IDF + LogisticRegression, 5-fold OOF",0.79,0.522
quality_unit,46,50,"TF-IDF + LogisticRegression, 4-fold OOF",0.625,0.947


In [8]:
pd.read_csv(P.outputs / 'c4_contract_clauses.csv')[['supplier_name', 'force_majeure', 'lead_time_days', 'late_delivery_penalty', 'exclusive_share_pct', 'contract_risk', 'llm_rule_agree_all_clauses']]

,supplier_name,force_majeure,lead_time_days,late_delivery_penalty,exclusive_share_pct,contract_risk,llm_rule_agree_all_clauses
0,"Accord Healthcare, Inc.",hormuz_disruption_excluded,45,weekly_ld_with_substitute_right,NaN,0.0625,True
1,Apotex Corp.,supplier_discretion_unlimited,60,ld_after_delay_threshold_low_cap,NaN,0.5000,True
2,"AVYXA Pharma, LLC",supplier_discretion_unlimited,60,ld_after_delay_threshold_low_cap,70.0,0.7500,True
3,BPI Labs LLC,balanced_capped,45,none,NaN,0.4375,True
4,Eugia US LLC,supplier_discretion_unlimited,75,none,80.0,0.9375,True
5,"Fresenius Kabi USA, LLC",balanced_capped,30,weekly_ld_with_substitute_right,NaN,0.1250,True
6,Gland Pharma Limited,hormuz_disruption_excluded,45,weekly_ld_with_substitute_right,NaN,0.0625,True
7,"Hospira, Inc.",balanced_capped,60,ld_after_delay_threshold_low_cap,NaN,0.3750,True
8,Natco Pharma USA LLC,balanced_capped,45,ld_after_delay_threshold_low_cap,60.0,0.5625,True
9,"Teva Parenteral Medicines, Inc.",balanced_capped,60,ld_after_delay_threshold_low_cap,NaN,0.3750,True


## The filled locked-schema matrix (rankable makers only)

In [9]:
mx = pd.read_csv(P.processed / 'suppliers_matrix.csv')
assert list(mx.columns[:16]) == pd.read_csv(P.l1_reference / 'locked_schema.csv')['column'].tolist()
mx[mx['eligible_for_ranking']][['manufacturer_id', 'is_probation', 'C1_landed_cost', 'C2_delivery_risk', 'C3_price_integrity', 'C4_compliance_risk', 'C5_route_exposure']]

,manufacturer_id,is_probation,C1_landed_cost,C2_delivery_risk,C3_price_integrity,C4_compliance_risk,C5_route_exposure
0,ACCORD HEALTHCARE,True,0.0,0.1033,0.0,0.6402,0.272
1,APOTEX,True,0.0,0.1033,0.0,0.1000,0.272
2,AVYXA PHARMA,True,1.0,0.0263,0.0,0.1000,0.153
3,BPI LABS,True,0.0,0.1033,0.0,0.0000,0.272
5,EUGIA US,True,0.0,0.1033,0.0,0.3000,0.272
6,FRESENIUS KABI USA,True,0.0,0.1033,0.0,0.3000,0.272
7,GLAND PHARMA,True,0.0,0.1033,0.0,0.0000,0.272
10,HOSPIRA,True,0.0,0.0681,0.0,0.3000,0.221
11,NATCO PHARMA USA,True,0.0,0.1033,0.0,0.1000,0.272
14,TEVA PARENTERAL MEDICINES,True,0.0,0.0289,0.0,0.3000,0.153
